In [1]:
import torch
import json
from pathlib import Path
import numpy as np
from models import build_model
from dataset_loader_safetensors import get_dataloaders

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


DATASET_DIR = "datasets"
BATCH_SIZE = 32

train_loader, val_loader, test_loader = get_dataloaders(
    DATASET_DIR,
    batch_size=BATCH_SIZE
)

In [3]:


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_PATH = "checkpoints/best_nnconv.pt"


sample_batch = next(iter(test_loader))
in_dim = sample_batch.x.shape[1]

model = build_model("nnconv", in_dim=in_dim).to(DEVICE)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

print("Model loaded successfully")

Model loaded successfully


In [4]:
@torch.no_grad()
def collect_predictions(loader):
    model.eval()

    all_preds, all_targets, all_pos = [], [], []

    for batch in loader:
        batch = batch.to(DEVICE)

        pred = model(batch)

        all_preds.append(pred.cpu())
        all_targets.append(batch.target.cpu())
        all_pos.append(batch.pos.cpu())

    return (
        torch.cat(all_preds, dim=0),
        torch.cat(all_targets, dim=0),
        torch.cat(all_pos, dim=0)
    )

In [5]:
preds, targets, positions = collect_predictions(test_loader)

In [6]:


errors = torch.norm(preds - targets, dim=1)

results = {
    "model": "nnconv",
    "mean_error": float(errors.mean()),
    "std_error": float(errors.std()),
    "median_error": float(errors.median()),
    "max_error": float(errors.max()),
    "rmse": float(torch.sqrt((errors ** 2).mean())),
}

In [7]:
from pathlib import Path
import json

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

with open(RESULTS_DIR / "nnconv_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

torch.save({
    "preds": preds,
    "targets": targets,
    "positions": positions
}, RESULTS_DIR / "nnconv_predictions.pt")

print("Saved evaluation results")

Saved evaluation results
